# Тестовая выборка D56 для метрики детекции (90 / 150 / 220 ГГц)

Ноутбук вызывает `scripts/make_test_set.py`: окна 480×480 с шагом 240 внутри поля D56 (138 на сетке),
одинаковые для всех частот; на каждое окно -- Planck, настоящий ACT и предсказание BBDM в FITS с WCS.

**Colab (по умолчанию).** Runtime -> GPU. Нужны:
- Drive с тайлами: `MyDrive/Galaxy_Hackers/Diffusion/{Planck/f100,f143,f217; Planck+ACT/f090,f150,f220}`;
- `runs.tar.gz` с обученными моделями (3 прогона `runs/d56_f90|150|220`), положенный в `MyDrive`
  (чужую папку: ПКМ -> «Добавить ярлык на Диск»; имя ярлыка задаёт `RUNS_ARCHIVE` ниже).

Выход: `OUT/f{90,150,220}/{planck,true,pred}/win_y{Y}_x{X}[_s{k}].fits`, `index.csv`, `manifest.json`;
в Colab готовый zip копируется на Drive (последняя ячейка). Окна перекрываются на 50%: детекции надо
дедуплицировать по координатам; 36 непересекающихся помечены `independent = 1` в `index.csv`.

### 0. Окружение (Colab)

In [ ]:
# Colab: подключить Drive, поставить пакеты, достать репозиторий. Локально / на сервере ничего не делает.
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "astropy", "scikit-image", "tqdm"], check=True)
    if os.path.isdir("/content/bbdm_cmb/.git"):
        subprocess.run(["git", "-C", "/content/bbdm_cmb", "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "https://github.com/Perf0rator4/bbdm-cmb.git",
                        "/content/bbdm_cmb"], check=True)
    print("Colab: Drive mounted, repo at /content/bbdm_cmb")
else:
    print("Skipped — running from a local checkout")

### 1. Пути и параметры

In [ ]:
import torch

if IN_COLAB:
    REPO = "/content/bbdm_cmb"
    DRIVE = "/content/drive/MyDrive"
    DATA_ROOT = f"{DRIVE}/Galaxy_Hackers/Diffusion"       # Planck/f100.., Planck+ACT/f090.. (как у train_freq.py)
    RUNS_ARCHIVE = f"{DRIVE}/runs.tar.gz"                 # ярлык / копия архива с прогонами
    OUT = "/content/test_set_d56"                         # локально (быстро); zip уйдёт на Drive в конце
    DRIVE_OUT = f"{DRIVE}/Galaxy_Hackers"
else:
    REPO = os.path.abspath("..")                          # ноутбук лежит в notebooks/
    DATA_ROOT = "/path/to/Diffusion"
    RUNS_ARCHIVE = None                                   # прогоны уже лежат в REPO/runs
    OUT = os.path.join(REPO, "test_set_d56")
    DRIVE_OUT = None

RUNS = [os.path.join(REPO, "runs", f"d56_f{f}") for f in (90, 150, 220)]

S = 200            # шагов сэмплера (как в оценке прогонов)
N_SAMPLES = 1      # сэмплов BBDM на окно (>1 -- для шумовой карты по ансамблю)
OVERWRITE = False  # True -- стереть прошлый набор в OUT и сделать заново
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, "scripts"))
print("device:", DEVICE)
if IN_COLAB and DEVICE == "cpu":
    print("!! Нет GPU: Runtime -> Change runtime type -> GPU. На CPU при S=200 это часы.")

### 1b. Модели (runs.tar.gz)

Из архива берутся только нужные файлы: `best_metric.pt`, `splits.json`, `results.json` (сам архив ~4,7 ГБ, чтение с Drive занимает несколько минут). Если `runs/` уже есть, ячейка ничего не делает.

In [ ]:
import tarfile

# best.pt не берём: у всех трёх прогонов есть best_metric.pt, который выбирает скрипт (+1,7 ГБ без пользы)
NEED = ("best_metric.pt", "splits.json", "results.json")


def extract_runs(archive, dest):
    # потоковое чтение: tar.gz не перематывается; .ipynb_checkpoints и last.pt пропускаем
    n = 0
    with tarfile.open(archive, "r|gz") as tf:
        for m in tf:
            if m.isfile() and "/.ipynb_checkpoints/" not in m.name and os.path.basename(m.name) in NEED:
                tf.extract(m, dest)
                n += 1
    return n


missing = [r for r in RUNS if not os.path.exists(os.path.join(r, "splits.json"))]
if not missing:
    print("runs/ уже на месте")
elif RUNS_ARCHIVE and os.path.exists(RUNS_ARCHIVE):
    print(f"извлекаю из {RUNS_ARCHIVE} ...")
    print("файлов:", extract_runs(RUNS_ARCHIVE, REPO))
else:
    raise FileNotFoundError(f"нет {missing}, и архива {RUNS_ARCHIVE!r} тоже: положите runs.tar.gz на Drive "
                            "(или распакуйте прогоны в REPO/runs)")

for r in RUNS:
    print(r, "OK" if os.path.exists(os.path.join(r, "splits.json")) else "НЕТ splits.json")

### 2. Нарезка и сэмплинг

На A40 при S=200 -- порядка десятков минут на все три частоты (оценка).

In [ ]:
import make_test_set

args = ["--data-root", DATA_ROOT, "--runs", *RUNS, "--out-dir", OUT,
        "--S", str(S), "--n-samples", str(N_SAMPLES), "--device", DEVICE]
make_test_set.main(args + (["--overwrite"] if OVERWRITE else []))

### 3. Что получилось

In [ ]:
import csv
import json

man = json.load(open(os.path.join(OUT, "manifest.json")))
index = list(csv.DictReader(open(os.path.join(OUT, "index.csv"))))
print(f"окон: {man['n_windows']} (непересекающихся {man['n_independent']}), отсев: {man['window_report']}")
for f, m in man["models"].items():
    print(f"{f} ГГц: {m['checkpoint']}  mu={m['mu']:.2f}  sigma={m['sigma']:.2f}")
for w in index[:3]:
    print(w)

### 4. Проверка: смещение среднего и разброс

На проверке с подставным небом предсказания всех трёх моделей имели постоянное смещение среднего
(−75 / −89 / −220 мкК). Спектральные метрики его не видят (бин 0 выброшен). Здесь -- на настоящих окнах.

In [ ]:
import numpy as np
from astropy.io import fits

def load(f, sub, name):
    return fits.getdata(os.path.join(OUT, f"f{f}", sub, name + ".fits")).astype(np.float64)

print(f"{'ГГц':>4} {'pred−true, мкК':>18} {'std pred':>9} {'std true':>9} {'corr':>6}")
for f in man["models"]:
    d, sp, st, r = [], [], [], []
    for name in (w["name"] for w in index):
        t, p = load(f, "true", name), load(f, "pred", name + "_s0")
        m = t != 0                                  # маска ACT
        d.append(p[m].mean() - t[m].mean())
        sp.append(p[m].std()); st.append(t[m].std())
        r.append(np.corrcoef(p[m], t[m])[0, 1])
    print(f"{f:>4} {np.mean(d):9.1f} ± {np.std(d):6.1f} {np.mean(sp):9.1f} {np.mean(st):9.1f} {np.mean(r):6.3f}")

### 5. Одно окно на всех частотах (общая шкала по true в каждой строке)

In [ ]:
import matplotlib.pyplot as plt

w = [w for w in index if w["independent"] == "1"][4]     # любое окно из index.csv
name = w["name"]
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
for row, f in zip(axes, man["models"]):
    t = load(f, "true", name)
    lim = np.percentile(np.abs(t[t != 0]), 99)
    for ax, (title, img) in zip(row, [("Planck", load(f, "planck", name)), ("ACT (true)", t),
                                      ("BBDM (pred)", load(f, "pred", name + "_s0"))]):
        ax.imshow(img, origin="lower", cmap="RdBu_r", vmin=-lim, vmax=lim)
        ax.set_title(f"{f} ГГц, {title}"); ax.axis("off")
fig.suptitle(f"{name}: RA {float(w['ra_center']):.2f}°, Dec {float(w['dec_center']):.2f}°")
plt.tight_layout()

### 6. Архив для Google Drive

Один zip (~1 ГБ) проще залить и передать, чем ~1200 отдельных FITS. В Colab он копируется в `MyDrive/Galaxy_Hackers/test_set_d56.zip`: дайте коллеге ссылку на этот файл.

In [ ]:
import shutil

archive = shutil.make_archive(OUT, "zip", root_dir=os.path.dirname(OUT), base_dir=os.path.basename(OUT))
print(archive, f"{os.path.getsize(archive) / 1e9:.2f} ГБ")
if DRIVE_OUT:
    shutil.copy(archive, DRIVE_OUT)
    print("скопировано на Drive:", os.path.join(DRIVE_OUT, os.path.basename(archive)))